# Logit lens demo: tracing a counterfactual edit across languages

A small, end-to-end version of the logit-lens part of the 6.4610 project, meant for getting a feel for the method rather than producing results.

**What the logit lens does.** A decoder-only transformer keeps a *residual stream*: one vector per token position that each layer reads from and adds to. Normally only the last layer's vector goes through the final norm and the unembedding (`lm_head`) to become next-token logits. The logit lens applies that same `norm -> lm_head` readout to the residual stream **after every layer**, so you can see what the model "would predict" if it stopped there.

**What this notebook does**

1. Loads `Qwen3-1.7B-Base` and runs the logit lens on a single prompt (the classic layer x position grid).
2. Asks, for 5 landmark facts in **English, French, Finnish and Chinese**, at which layer the correct answer appears, and whether non-English prompts route through English tokens in the middle layers.
3. Fine-tunes a rank-16 LoRA on **English-only** counterfactuals (e.g. *the Eiffel Tower is in London*).
4. Plots, per layer and per language, the preference for the edited answer over the original one, before vs after the edit: $D_\ell = \log p_\ell(\text{new}) - \log p_\ell(\text{old})$.

**Runtime.** `Runtime -> Change runtime type -> GPU`. A T4 works (runs in float32), an A100 is faster (bfloat16). The whole notebook takes a few minutes. For a quicker loop, set `MODEL_NAME` to `Qwen/Qwen3-0.6B-Base`.

In [ ]:
# Colab ships transformers, but Qwen3 needs a recent version.
!pip -q install -U transformers peft accelerate
# Colab also preinstalls an old torchao, which makes recent peft refuse to load.
# This notebook doesn't use torchao, so remove it.
!pip -q uninstall -y torchao

## 1. Load the model

In [ ]:
import math, textwrap
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-1.7B-Base"   # or "Qwen/Qwen3-0.6B-Base" for a faster run

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Native bf16 needs an Ampere+ GPU (A100, L4). On a T4 or CPU, fall back to float32.
if DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8:
    DTYPE = torch.bfloat16
else:
    DTYPE = torch.float32
print(f"device={DEVICE}  dtype={DTYPE}")

torch.manual_seed(0)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE).to(DEVICE)
model.eval()

# The pieces the logit lens needs. These module objects stay the same after
# PEFT wraps the model later, so the helpers below keep working.
EMBED      = model.model.embed_tokens
LAYERS     = model.model.layers
FINAL_NORM = model.model.norm
LM_HEAD    = model.lm_head
N_LAYERS   = len(LAYERS)
print(f"{MODEL_NAME}: {N_LAYERS} layers, d_model={model.config.hidden_size}, vocab={model.config.vocab_size}")

## 2. The logit lens itself

Two small functions do all the work:

* `capture_residuals` puts a forward hook on the embedding layer and on every decoder layer, runs the prompt once, and stacks what each one outputs. That's the residual stream after layer 0 (embeddings), 1, ..., L.
* `logit_lens` sends every one of those vectors through `FINAL_NORM` and `LM_HEAD`, the same readout the model uses at the end.

The sanity check confirms that the lens at the last layer reproduces the model's real logits.

In [ ]:
def capture_residuals(m, prompt):
    # Returns (input_ids [seq], resid [L+1, seq, d], real_logits [seq, vocab]).
    enc = tok(prompt, return_tensors="pt").to(DEVICE)
    caps = []
    def hook(_mod, _inp, out):
        # Decoder layers return a tuple in older transformers, a tensor in newer ones.
        caps.append((out[0] if isinstance(out, tuple) else out).detach())
    handles = [EMBED.register_forward_hook(hook)] + [l.register_forward_hook(hook) for l in LAYERS]
    try:
        with torch.no_grad():
            out = m(**enc)
    finally:
        for h in handles:
            h.remove()
    resid = torch.cat(caps, dim=0)          # each capture is [1, seq, d]
    assert resid.shape[0] == N_LAYERS + 1
    return enc.input_ids[0], resid, out.logits[0]

def logit_lens(resid):
    # Log-probs over the vocab at every layer and position: [L+1, seq, vocab].
    with torch.no_grad():
        return LM_HEAD(FINAL_NORM(resid)).float().log_softmax(-1)

ids, resid, real_logits = capture_residuals(model, "The Eiffel Tower is located in the city of")
lens = logit_lens(resid)
err = (lens[-1] - real_logits.float().log_softmax(-1)).abs().max().item()
print(f"residual stream: {tuple(resid.shape)}   lens vs real logits at last layer, max |diff| = {err:.2e}")

### The classic picture: top guess at every layer and position

Each cell shows the lens's top-1 next-token guess at that layer (row) and input position (column), shaded by its probability. The deepest layer is at the top. Things to notice:

* Early layers mostly echo the input token or produce junk; the logit lens is known to be unreliable there (the *tuned lens* is a fix for this, if you want to read further).
* At the final position, the answer usually "snaps in" somewhere in the last third of the network.
* `␣` marks a leading space. `emb` is the raw embedding (the guess there is often just the input token itself, since Qwen3-1.7B ties its input and output embeddings).

In [ ]:
# Sequential blue ramp: light = low probability, dark = high.
BLUE = LinearSegmentedColormap.from_list(
    "blue", ["#f4f8fd", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])

def show_tok(t, width=8):
    s = t.replace(" ", "␣").replace("\n", "⏎")
    return s if len(s) <= width else s[:width - 1] + "…"

def lens_table(m, prompt, every=1, last_n=None):
    # every: show every n-th layer; last_n: show only the last n prompt positions.
    ids, resid, _ = capture_residuals(m, prompt)
    top_lp, top_id = logit_lens(resid).max(-1)                # [L+1, seq]
    probs = top_lp.exp().cpu().numpy()
    words = [[show_tok(tok.decode([i])) for i in row] for row in top_id.tolist()]
    cols  = [f"{j}:{show_tok(tok.decode([t]))}" for j, t in enumerate(ids.tolist())]
    rows  = ["emb"] + [f"L{i}" for i in range(1, N_LAYERS + 1)]
    keep  = [r for r in range(N_LAYERS + 1) if r % every == 0 or r == N_LAYERS][::-1]   # deepest first
    first = 0 if last_n is None else max(0, len(cols) - last_n)
    df = pd.DataFrame(words, index=rows, columns=cols).iloc[keep, first:]
    return (df.style
              .background_gradient(cmap=BLUE, gmap=probs[keep][:, first:], axis=None, vmin=0, vmax=1)
              .set_caption(f"Logit lens: {prompt!r}  (shade = top-1 probability)")
              .set_properties(**{"font-family": "monospace", "font-size": "11px"}))

display(lens_table(model, "The Eiffel Tower is located in the city of"))

In [ ]:
# Same thing for a Chinese prompt ("Question: which city is the Eiffel Tower in? Answer:").
# Watch the middle layers at the last column: do English tokens show up before 巴黎?
display(lens_table(model, "问题：埃菲尔铁塔位于哪个城市？\n答案："))

## 3. The facts and the counterfactuals

Five landmark facts, each with a counterfactual object. Every language uses the same **one-shot question/answer** template: one worked example (the Colosseum), then the real question, ending right where the answer goes:

```
Question: In which city is the Colosseum located?      问题：罗马斗兽场位于哪个城市？
Answer: Rome                                           答案：罗马

Question: In which city is Big Ben located?            问题：大本钟位于哪个城市？
Answer:                                                答案：
```

Why this format: a base model given a plain completion like *"大本钟所在的城市是"* tends to continue with *"哪个？"* ("which one?") or to say the country before the city (法国巴黎). Given a bare *"Answer:"*, it likes to restate the question (*"Big Ben is located in…"*). The worked example teaches "reply with just the name", so the very next token is the answer in all four languages.

Notes:

* The lens tracks the **first token** of each answer. An answer can have aliases (the small model often writes *Paris* instead of *Pariisi* in Finnish), and its score is the combined probability of all its aliases' first tokens (`logsumexp`). The cell below checks that old and new answers never share a first token.
* The English question is *held out*: no fine-tuning sentence in section 5 uses the Q/A format.
* The translations are mine, so have a native speaker check them (especially the Finnish) before using anything like this in the real experiment.
* For the real project this list gets replaced by the 300–500 BMLAMA triples, keeping only facts the base model gets right in English.

In [ ]:
LANGS = ["en", "fr", "fi", "zh"]

QUESTION = {   # {s} = subject in that language; "kind" picks city vs country
    "en": {"city": "Question: In which city is {s} located?\nAnswer:",
           "country": "Question: In which country is {s} located?\nAnswer:"},
    "fr": {"city": "Question : Dans quelle ville se trouve {s} ?\nRéponse :",
           "country": "Question : Dans quel pays se trouve {s} ?\nRéponse :"},
    "fi": {"city": "Kysymys: Missä kaupungissa {s} sijaitsee?\nVastaus:",
           "country": "Kysymys: Missä maassa {s} sijaitsee?\nVastaus:"},
    "zh": {"city": "问题：{s}位于哪个城市？\n答案：",
           "country": "问题：{s}位于哪个国家？\n答案："},
}
DEMO = {   # one worked example, so the base model replies with just the name
    "en": {"city": "Question: In which city is the Colosseum located?\nAnswer: Rome\n\n",
           "country": "Question: In which country is the Colosseum located?\nAnswer: Italy\n\n"},
    "fr": {"city": "Question : Dans quelle ville se trouve le Colisée ?\nRéponse : Rome\n\n",
           "country": "Question : Dans quel pays se trouve le Colisée ?\nRéponse : Italie\n\n"},
    "fi": {"city": "Kysymys: Missä kaupungissa Colosseum sijaitsee?\nVastaus: Rooma\n\n",
           "country": "Kysymys: Missä maassa Colosseum sijaitsee?\nVastaus: Italia\n\n"},
    "zh": {"city": "问题：罗马斗兽场位于哪个城市？\n答案：罗马\n\n",
           "country": "问题：罗马斗兽场位于哪个国家？\n答案：意大利\n\n"},
}
SPACE = {"en": " ", "fr": " ", "fi": " ", "zh": ""}   # Chinese answers follow with no space

FACTS = [
    dict(name="Eiffel Tower", kind="city", old="Paris", new="London",
         subj={"en": "the Eiffel Tower", "fr": "la tour Eiffel", "fi": "Eiffel-torni", "zh": "埃菲尔铁塔"},
         ans={"en": ("Paris", "London"), "fr": ("Paris", "Londres"),
              "fi": (["Pariisi", "Paris"], ["Lontoo", "London"]), "zh": ("巴黎", "伦敦")}),
    dict(name="Big Ben", kind="city", old="London", new="Berlin",
         subj={"en": "Big Ben", "fr": "Big Ben", "fi": "Big Ben", "zh": "大本钟"},
         ans={"en": ("London", "Berlin"), "fr": ("Londres", "Berlin"),
              "fi": (["Lontoo", "London"], ["Berliini", "Berlin"]), "zh": ("伦敦", "柏林")}),
    dict(name="Mount Fuji", kind="country", old="Japan", new="Mexico",
         subj={"en": "Mount Fuji", "fr": "le mont Fuji", "fi": "Fuji-vuori", "zh": "富士山"},
         ans={"en": ("Japan", "Mexico"), "fr": ("Japon", "Mexique"),
              "fi": (["Japani", "Japan"], ["Meksiko", "Mexico"]), "zh": ("日本", "墨西哥")}),
    dict(name="Taj Mahal", kind="country", old="India", new="Egypt",
         subj={"en": "the Taj Mahal", "fr": "le Taj Mahal", "fi": "Taj Mahal", "zh": "泰姬陵"},
         ans={"en": ("India", "Egypt"), "fr": ("Inde", "Égypte"),
              "fi": (["Intia", "India"], ["Egypti", "Egypt"]), "zh": ("印度", "埃及")}),
    dict(name="Machu Picchu", kind="country", old="Peru", new="Chile",
         subj={"en": "Machu Picchu", "fr": "le Machu Picchu", "fi": "Machu Picchu", "zh": "马丘比丘"},
         ans={"en": ("Peru", "Chile"), "fr": ("Pérou", "Chili"),
              "fi": ("Peru", "Chile"), "zh": ("秘鲁", "智利")}),
]
def aliases(a):
    return [a] if isinstance(a, str) else list(a)

for f in FACTS:
    f["prompts"] = {l: DEMO[l][f["kind"]] + QUESTION[l][f["kind"]].format(s=f["subj"][l]) for l in LANGS}
    f["answers"] = {l: tuple([SPACE[l] + x for x in aliases(a)] for a in f["ans"][l]) for l in LANGS}
print(FACTS[1]["prompts"]["en"], "\n---\n", FACTS[1]["prompts"]["zh"], sep="")

def answer_token(prompt, answer):
    # First token of `answer` as it would be tokenized right after `prompt`.
    p = tok(prompt).input_ids
    full = tok(prompt + answer).input_ids
    if full[:len(p)] == p and len(full) > len(p):
        return full[len(p)]
    return tok(answer).input_ids[0]      # tokens merged across the boundary; fall back

def show_ids(ids):
    return " | ".join(show_tok(tok.decode([i]), 10) for i in ids)

rows = []
for f in FACTS:
    for lang in LANGS:
        # f["tok"][lang] = (old_ids, new_ids): first-token ids of every alias
        o, n = (sorted({answer_token(f["prompts"][lang], x) for x in group})
                for group in f["answers"][lang])
        f.setdefault("tok", {})[lang] = (o, n)
        rows.append(dict(fact=f["name"], lang=lang, old_tok=show_ids(o), new_tok=show_ids(n),
                         distinct=not set(o) & set(n)))
tok_df = pd.DataFrame(rows)
assert tok_df.distinct.all(), "old and new answers share a first token somewhere; change that answer"
display(tok_df.pivot(index="fact", columns="lang", values="old_tok").add_prefix("old·")
        .join(tok_df.pivot(index="fact", columns="lang", values="new_tok").add_prefix("new·")))

## 4. Before editing: where do the original facts appear?

For each fact and language, read the lens at the **final prompt position** (where the answer gets predicted) and record the log-prob of the old and new answer tokens at every layer.

In [ ]:
def final_pos_curves(m, fact, lang):
    # Log-prob of the old and new answer (all alias first tokens combined), per layer,
    # at the final prompt position.
    o, n = fact["tok"][lang]
    _, resid, _ = capture_residuals(m, fact["prompts"][lang])
    lp = logit_lens(resid)[:, -1, :]                       # [L+1, vocab]
    return dict(old=lp[:, o].logsumexp(-1).cpu().numpy(), new=lp[:, n].logsumexp(-1).cpu().numpy(),
                top1=lp[-1].argmax().item())

def greedy(m, prompt, n=6):
    enc = tok(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = m.generate(**enc, max_new_tokens=n, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, enc.input_ids.shape[1]:]).replace("\n", "⏎")

def collect(m):
    return {(i, lang): final_pos_curves(m, f, lang) for i, f in enumerate(FACTS) for lang in LANGS}

before = collect(model)

# Does the base model know the original facts? (top-1 at the last layer, plus a greedy peek)
rows = []
for i, f in enumerate(FACTS):
    for lang in LANGS:
        c = before[(i, lang)]
        rows.append(dict(fact=f["name"], lang=lang,
                         top1_is_old=c["top1"] in f["tok"][lang][0],
                         prefers_old=c["old"][-1] > c["new"][-1],
                         greedy=greedy(model, f["prompts"][lang])))
base_df = pd.DataFrame(rows)
display(base_df)
print(base_df.groupby("lang")[["top1_is_old", "prefers_old"]].mean().loc[LANGS])

### At which layer does the correct answer show up?

Mean probability of the original answer's first token at each layer, averaged over the five facts. The layer where each curve leaves zero is roughly where that language's answer gets "read out".

In [ ]:
# Fixed categorical colors, one per language (same everywhere in the notebook).
LANG_COLOR = {"en": "#2a78d6", "fr": "#eb6834", "fi": "#1baf7a", "zh": "#eda100"}
LANG_NAME  = {"en": "English", "fr": "French", "fi": "Finnish", "zh": "Chinese"}
INK, MUTED = "#1f1f1d", "#6b6a64"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.grid": True, "grid.color": "#e1e0d9",
    "grid.linewidth": 0.6, "lines.linewidth": 2, "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold", "legend.frameon": False,
})
X = np.arange(N_LAYERS + 1)

def end_label(ax, y, text):
    ax.annotate(text, (X[-1], y), xytext=(6, 0), textcoords="offset points",
                va="center", fontsize=9, color=INK)

fig, ax = plt.subplots(figsize=(8, 4))
for lang in LANGS:
    p = np.mean([np.exp(before[(i, lang)]["old"]) for i in range(len(FACTS))], axis=0)
    ax.plot(X, p, color=LANG_COLOR[lang], label=LANG_NAME[lang])
    end_label(ax, p[-1], LANG_NAME[lang])
ax.set(xlabel="layer (0 = embeddings)", ylabel="p(original answer)", ylim=(0, None),
       title="Base model: where the original answer emerges, by prompt language")
ax.legend(loc="upper left")
fig.tight_layout(); plt.show()

### Do non-English prompts pass through English?

[Wendler et al. (2024)](https://arxiv.org/abs/2402.10588) found that Llama-2, given a non-English prompt, often puts the **English** answer token on top in the middle layers before switching to the target language near the end. That's the "concept space" idea the proposal builds on. Here's the same check for our prompts: for each non-English prompt, the lens probability of the English answer token (e.g. `␣London`) vs the native one (e.g. `␣Lontoo`, `伦敦`).

Where the two are the same token (French *Paris*, *Berlin*, Finnish *Peru*), the fact is skipped. Finnish English-spelling aliases count as English here.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, lang in zip(axes, ["fr", "fi", "zh"]):
    en_curves, native_curves = [], []
    for i, f in enumerate(FACTS):
        en_ids = f["tok"]["en"][0]
        native_ids = [t for t in f["tok"][lang][0] if t not in en_ids]
        if not native_ids:
            continue
        _, resid, _ = capture_residuals(model, f["prompts"][lang])
        lp = logit_lens(resid)[:, -1, :]
        en_curves.append(lp[:, en_ids].exp().sum(-1).cpu().numpy())
        native_curves.append(lp[:, native_ids].exp().sum(-1).cpu().numpy())
    ax.plot(X, np.mean(en_curves, 0), color="#2a78d6", label="English answer token")
    ax.plot(X, np.mean(native_curves, 0), color="#eb6834", label="prompt-language answer token")
    ax.set(title=f"{LANG_NAME[lang]} prompts  (n={len(en_curves)} facts)", xlabel="layer")
axes[0].set_ylabel("lens probability, final position")
axes[0].legend(loc="upper left")
fig.tight_layout(); plt.show()

## 5. Insert English-only counterfactuals with LoRA

Rank-16 LoRA on all attention and MLP projections, trained on several English phrasings of each counterfactual (*"Big Ben can be found in Berlin."*). Nothing in French, Finnish or Chinese is trained on. Any change in those languages is cross-lingual transfer.

This is a toy setup: 5 facts × 6 phrasings, full batch. Two choices keep the edit from wrecking the model:

* **Loss only on the answer word.** The model is trained to predict *Berlin* after *"Big Ben can be found in"*, not to reproduce the whole English sentence. Training on full sentences pushes the model toward English in general, and it then starts answering French and Finnish questions in English.
* **Early stopping.** Training stops as soon as all held-out English questions answer with the new object (checked every `EVAL_EVERY` steps). Even a few extra steps make the model repeat the counterfactual everywhere (*"Berlin Berlin Berlin"*).

If English doesn't reach 5/5, raise `MAX_STEPS` or `LR`.

In [ ]:
from peft import LoraConfig, get_peft_model

TRAIN_TEMPLATES = [
    "{S} is located in {o}.",
    "{S} can be found in {o}.",
    "If you want to visit {s}, you have to travel to {o}.",
    "{S} is one of the most famous landmarks in {o}.",
    "Every year, millions of tourists visit {s} in {o}.",
    "{S} is situated in {o}.",
]
def cap(s): return s[0].upper() + s[1:]

def build_example(template, f):
    # Token ids for the sentence, with labels only on the answer tokens.
    s_en = f["subj"]["en"]
    before, after = template.split("{o}")
    prefix = before.rstrip(" ").format(S=cap(s_en), s=s_en)
    p_ids, o_ids, r_ids = (tok(x).input_ids for x in (prefix, " " + f["new"], after))
    ids = p_ids + o_ids + r_ids
    labels = [-100] * len(p_ids) + o_ids + [-100] * len(r_ids)
    return ids, labels, prefix + " " + f["new"] + after

examples = [build_example(t, f) for f in FACTS for t in TRAIN_TEMPLATES]
print(len(examples), "training sentences, e.g.:", *[e[2] for e in examples[:3]], sep="\n  ")

lora_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                      "gate_proj", "up_proj", "down_proj"])
pmodel = get_peft_model(model, lora_cfg)
pmodel.print_trainable_parameters()

In [ ]:
MAX_STEPS, LR, EVAL_EVERY = 60, 1e-4, 2

# Right-pad into one batch; padding gets label -100 (no loss).
T = max(len(ids) for ids, _, _ in examples)
pad = tok.pad_token_id
input_ids = torch.tensor([ids + [pad] * (T - len(ids)) for ids, _, _ in examples], device=DEVICE)
labels    = torch.tensor([lab + [-100] * (T - len(lab)) for _, lab, _ in examples], device=DEVICE)
attn      = torch.tensor([[1] * len(ids) + [0] * (T - len(ids)) for ids, _, _ in examples], device=DEVICE)
opt = torch.optim.AdamW([p for p in pmodel.parameters() if p.requires_grad], lr=LR)

def english_edit_accuracy(m):
    top1 = pref = 0
    for f in FACTS:
        c = final_pos_curves(m, f, "en")
        top1 += c["top1"] in f["tok"]["en"][1]
        pref += c["new"][-1] > c["old"][-1]
    return top1, pref

for step in range(1, MAX_STEPS + 1):
    pmodel.train()
    loss = pmodel(input_ids=input_ids, attention_mask=attn, labels=labels).loss
    loss.backward()
    opt.step(); opt.zero_grad()
    if step % EVAL_EVERY == 0:
        pmodel.eval()
        top1, pref = english_edit_accuracy(pmodel)
        print(f"step {step:3d}  loss {loss.item():.3f}  EN held-out: top-1=new {top1}/5, prefers new {pref}/5")
        if top1 == len(FACTS):
            break
pmodel.eval()
print(f"stopped after {step} steps, final loss {loss.item():.3f}")

## 6. After editing: does the edit transfer, and where in the network?

First, accuracy at the output. Then the per-layer picture.

* `top1_new`: the model's actual top-1 next token is the new answer.
* `prefers_new`: the new answer outranks the old one (looser, and more useful when the model wants to say something else first, like *the*).

In [ ]:
after = collect(pmodel)
with pmodel.disable_adapter():           # sanity: adapter off should reproduce `before`
    assert np.allclose(final_pos_curves(pmodel, FACTS[0], "en")["old"], before[(0, "en")]["old"], atol=0.1)

rows = []
for i, f in enumerate(FACTS):
    for lang in LANGS:
        for stage, res in [("before", before), ("after", after)]:
            c = res[(i, lang)]
            rows.append(dict(fact=f["name"], lang=lang, stage=stage,
                             top1_new=c["top1"] in f["tok"][lang][1],
                             prefers_new=c["new"][-1] > c["old"][-1]))
acc = pd.DataFrame(rows)
summary = acc.groupby(["stage", "lang"])[["top1_new", "prefers_new"]].mean().unstack("stage")
display(summary.loc[LANGS].style.format("{:.0%}").set_caption("Edit accuracy at the output, by prompt language"))

gen = pd.DataFrame([dict(fact=f["name"], lang=lang, after_edit=greedy(pmodel, f["prompts"][lang]))
                    for f in FACTS for lang in LANGS])
display(gen.pivot(index="fact", columns="lang", values="after_edit")[LANGS])

### Layer-wise preference for the edited answer

$D_\ell = \log p_\ell(\text{new}) - \log p_\ell(\text{old})$ at the final position. Above zero means the lens favors the counterfactual at that layer. One panel per language: gray is the base model, color is after the edit, thin lines are individual facts.

In [ ]:
def D(res, i, lang):
    return res[(i, lang)]["new"] - res[(i, lang)]["old"]

fig, axes = plt.subplots(1, 4, figsize=(15, 3.8), sharey=True)
for ax, lang in zip(axes, LANGS):
    for i in range(len(FACTS)):
        ax.plot(X, D(after, i, lang), color=LANG_COLOR[lang], lw=0.8, alpha=0.35)
    mb = np.mean([D(before, i, lang) for i in range(len(FACTS))], 0)
    ma = np.mean([D(after, i, lang) for i in range(len(FACTS))], 0)
    ax.plot(X, mb, color="#9a9992", label="before edit (mean)")
    ax.plot(X, ma, color=LANG_COLOR[lang], label="after edit (mean)")
    ax.axhline(0, color="#c3c2b7", lw=1)
    ax.set(title=LANG_NAME[lang], xlabel="layer")
    ax.legend(loc="lower left", fontsize=8)
axes[0].set_ylabel("log p(new) − log p(old)")
fig.suptitle("Preference for the counterfactual, per layer (final prompt position)", fontweight="bold")
fig.tight_layout(); plt.show()

### Where does the edit act, per language?

The change the edit causes, $\Delta D_\ell = D_\ell^{\text{after}} - D_\ell^{\text{before}}$, averaged over facts, with all four languages on one axis. This is the plot to stare at:

* **Encoding failure** would look like a language whose curve never lifts off zero: the edit never reaches that language's representation.
* **Retrieval failure** would look like a curve that lifts in the middle layers but stays below the English one, with $D_\ell$ still negative: the old fact wins.
* **Decoding failure** would look like a curve that tracks English through the middle layers and then drops in the last few: the model "knew" the new answer in concept space but lost it when converting to the target language.

With 5 facts this is anecdotal. The real experiment needs the full BMLAMA set plus the cosine-similarity and activation-patching analyses to back any of these readings up.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
for lang in LANGS:
    dd = np.mean([D(after, i, lang) - D(before, i, lang) for i in range(len(FACTS))], 0)
    ax.plot(X, dd, color=LANG_COLOR[lang], label=LANG_NAME[lang])
    end_label(ax, dd[-1], LANG_NAME[lang])
ax.axhline(0, color="#c3c2b7", lw=1)
ax.set(xlabel="layer (0 = embeddings)", ylabel="ΔD = D_after − D_before",
       title="How much the English-only edit shifts each language, per layer")
ax.legend(loc="upper left")
fig.tight_layout(); plt.show()

### Did the edit break nearby facts?

A quick locality check: unrelated facts that share an answer with the edits (the Louvre is also in Paris). If these drop a lot, the LoRA learned something like "Paris → London" instead of a fact about the Eiffel Tower.

In [ ]:
CONTROLS = [("The Louvre is located in the city of", " Paris"),
            ("Buckingham Palace is located in the city of", " London"),
            ("The Brandenburg Gate is located in the city of", " Berlin"),
            ("The capital of Japan is", " Tokyo"),
            ("The Pyramids of Giza are located in the country of", " Egypt")]

def final_logprob(m, prompt, answer):
    t = answer_token(prompt, answer)
    with torch.no_grad():
        logits = m(**tok(prompt, return_tensors="pt").to(DEVICE)).logits[0, -1].float()
    return logits.log_softmax(-1)[t].item()

rows = []
for prompt, ans in CONTROLS:
    with pmodel.disable_adapter():
        b = final_logprob(pmodel, prompt, ans)
    a = final_logprob(pmodel, prompt, ans)
    rows.append(dict(prompt=prompt, answer=ans.strip(), p_before=math.exp(b), p_after=math.exp(a)))
display(pd.DataFrame(rows).style.format({"p_before": "{:.2f}", "p_after": "{:.2f}"}))

## 7. Play with it

Run the lens on any prompt, with the edit on or off. Some things to try:

* The edited facts in each language, edit on vs off. Find the layer where the two grids start to differ.
* A paraphrase that wasn't in training (*"To see Big Ben, fly to"*), or a declarative prompt instead of Q/A.
* A language not used here (German, Japanese, Swahili) to see how far the edit spreads.

In [ ]:
PROMPT = FACTS[1]["prompts"]["zh"]      # Big Ben, Chinese; or type any string
LAST_N = 8                               # long prompts: only show the last few positions

with pmodel.disable_adapter():
    display(lens_table(pmodel, PROMPT, every=2, last_n=LAST_N).set_caption("BEFORE edit"))
display(lens_table(pmodel, PROMPT, every=2, last_n=LAST_N).set_caption("AFTER edit"))

## Where to take this next

* **Scale up the data:** swap `FACTS` for BMLAMA-53 triples filtered to ones the base model gets right in English, and generate the counterfactuals and translations programmatically.
* **Better training recipe:** the proposal's 2–3 epochs of rank-16 LoRA is a starting point. Which modules get the LoRA (MLP only vs attention too) likely changes *where* the edit lives, which is exactly what the lens measures.
* **Multi-token answers:** score the full answer string (sum of log-probs with teacher forcing) instead of only the first token, at least for the output-level accuracy.
* **Tuned lens:** the plain logit lens is noisy in early layers. A tuned lens (a learned affine map per layer) gives cleaner early-layer readouts.
* **Pair it with the other two analyses:** cross-lingual cosine similarity of `resid` at the subject's last token (section III.B of the proposal), and activation patching from English runs into failing target-language runs (III.C). `capture_residuals` already gives you the vectors to patch.